# 01 Data Acquisition And Validation

This notebook documents the reproducible acquisition and initial validation workflow for the EIA-861M electricity retail sales dataset.

The notebook is intentionally not the main software architecture. Reusable logic lives in `src/eia861m/`, while this notebook acts as a transparent analytical interface for inspection, validation, and explanation.

## Scope

Project 01 focuses on Business Analytics and Decision Intelligence; forecasting is reserved for Project 02. At this stage, the objective is not EDA or dashboarding. The objective is to make sure the dataset is traceable, structurally valid, and safe to use for downstream analysis.

Key documentation:

- `../docs/data_provenance.md`
- `../docs/data_quality.md`
- `../docs/data_dictionary.md`

## Data Handling Rules

- Treat raw API data as reproducible source data.
- Do not silently overwrite or manually edit raw data.
- Retain published zero values unless a documented source justifies different treatment.
- Do not interpret `TRA` as all transportation activity.
- Use quality flags for investigation instead of deleting surprising observations.

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

# Colab convenience: mount Google Drive when this notebook runs in Colab.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    pass

# Recommended Drive location for this project.
RECOMMENDED_COLAB_ROOT = Path(
    "/content/drive/MyDrive/ds_portfolio/project_01_eia861m_electricity_sales"
)

# If auto-detection fails in Colab/Jupyter, set this manually to the project folder.
# Example:
# PROJECT_ROOT_OVERRIDE = Path("/content/drive/MyDrive/ds_portfolio/project_01_eia861m_electricity_sales")
PROJECT_ROOT_OVERRIDE = RECOMMENDED_COLAB_ROOT if RECOMMENDED_COLAB_ROOT.exists() else None


def find_project_root() -> Path:
    """Locate the project root containing src/eia861m and configs/config.json."""
    candidates = []

    env_root = os.getenv("EIA861M_PROJECT_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())

    if PROJECT_ROOT_OVERRIDE is not None:
        candidates.append(Path(PROJECT_ROOT_OVERRIDE).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    for candidate in candidates:
        if (candidate / "src" / "eia861m").exists() and (candidate / "configs" / "config.json").exists():
            return candidate.resolve()

    searched = "\n".join(str(candidate) for candidate in candidates)
    raise RuntimeError(
        "Could not locate the project root. This notebook must be run with the full project folder available, "
        "not as a standalone uploaded notebook.\n\n"
        "Fix options:\n"
        "1. Upload/sync the full folder to MyDrive/ds_portfolio/project_01_eia861m_electricity_sales.\n"
        "2. Set PROJECT_ROOT_OVERRIDE manually to the project folder.\n"
        "3. Set environment variable EIA861M_PROJECT_ROOT to the project folder.\n\n"
        f"Current working directory: {cwd}\n\n"
        f"Searched candidate paths:\n{searched}"
    )


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from eia861m.config import load_config
from eia861m.data_acquisition import fetch_dataset
from eia861m.env import load_env_file
from eia861m.paths import project_path
from eia861m.validation import (
    NUMERIC_COLUMNS,
    add_quality_flags,
    build_validation_summary,
)

print(f"Project root: {PROJECT_ROOT}")

## Configuration

The project configuration defines the API endpoint, requested fields, date windows, states, sectors, output paths, and validation expectations. Keeping these values in `configs/config.json` prevents important parameters from being scattered across notebook cells.

In [ ]:
config = load_config()

scope_summary = pd.DataFrame(
    [
        {"item": "start_period", "value": config["dataset"]["start_period"]},
        {"item": "end_period", "value": config["dataset"]["end_period"]},
        {"item": "states", "value": len(config["dataset"]["states"])},
        {"item": "sectors", "value": ", ".join(config["dataset"]["sectors"])},
        {"item": "request_windows", "value": len(config["dataset"]["request_windows"])},
        {"item": "expected_rows", "value": config["validation"]["expected_rows"]},
    ]
)

scope_summary

## Acquisition

If the raw CSV already exists, this notebook loads it. If it does not exist, the notebook attempts to fetch the data from the EIA API using `EIA_API_KEY` from the environment or a local `.env` file.

The raw-data policy is to regenerate data through scriptable acquisition rather than depend on an undocumented manual download.

In [ ]:
load_env_file(project_path(".env"))

raw_path = project_path(config["paths"]["raw_data"])
metadata_path = project_path(config["paths"]["acquisition_metadata"])
api_key = os.getenv("EIA_API_KEY")

if raw_path.exists():
    df_raw = pd.read_csv(raw_path)
    acquisition_metadata = (
        json.loads(metadata_path.read_text(encoding="utf-8"))
        if metadata_path.exists()
        else {"source": "existing raw CSV", "metadata_file_found": False}
    )
    print(f"Loaded existing raw data: {raw_path}")
else:
    if not api_key:
        raise RuntimeError(
            "Raw data is not available and EIA_API_KEY is not set. "
            "Set EIA_API_KEY from Colab Secrets before this cell, or use a private local .env outside shared folders."
        )

    df_raw, acquisition_metadata = fetch_dataset(config=config, api_key=api_key)
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    metadata_path.parent.mkdir(parents=True, exist_ok=True)
    with raw_path.open("w", encoding="utf-8", newline="") as file:
        df_raw.to_csv(file, index=False, lineterminator="\n")
    metadata_path.write_bytes(json.dumps(acquisition_metadata, indent=2).encode("utf-8"))
    print(f"Downloaded and saved raw data: {raw_path}")

print(f"Rows: {len(df_raw):,}")
print(f"Columns: {len(df_raw.columns):,}")

In [ ]:
df_raw.head()

In [ ]:
df_raw.dtypes

## Structural Validation

The expected key is `period + stateid + sectorid`. The dataset should contain one row per state-month-sector combination.

In [ ]:
validation_summary = build_validation_summary(df_raw, config=config)

pd.DataFrame(
    [
        {"check": "row_count", "observed": validation_summary["row_count"], "expected": validation_summary["expected"]["rows"]},
        {"check": "unique_months", "observed": validation_summary["unique_months"], "expected": validation_summary["expected"]["months"]},
        {"check": "unique_states", "observed": validation_summary["unique_states"], "expected": validation_summary["expected"]["states"]},
        {"check": "unique_sectors", "observed": validation_summary["unique_sectors"], "expected": validation_summary["expected"]["sectors"]},
        {"check": "duplicate_key_count", "observed": validation_summary["duplicate_key_count"], "expected": 0},
    ]
)

In [ ]:
pd.DataFrame(
    {
        "missing_count": validation_summary["missing_counts"],
    }
).sort_index()

## Quality Flags

Quality flags support investigation without silently changing the source data. They are not deletion rules.

In [ ]:
df_flagged = add_quality_flags(
    df_raw,
    revenue_abs_tolerance_million_usd=float(
        config["validation"]["revenue_abs_tolerance_million_usd"]
    ),
)

flagged_path = project_path(config["paths"]["validation_flags"])
summary_path = project_path(config["paths"]["validation_summary"])
flagged_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.parent.mkdir(parents=True, exist_ok=True)

with flagged_path.open("w", encoding="utf-8", newline="") as file:
    df_flagged.to_csv(file, index=False, lineterminator="\n")
summary_path.write_bytes(json.dumps(validation_summary, indent=2).encode("utf-8"))

print(f"Saved flagged validation dataset: {flagged_path}")
print(f"Saved validation summary: {summary_path}")

In [ ]:
flag_columns = [column for column in df_flagged.columns if column.startswith("flag_")]

pd.DataFrame(
    {
        "true_count": df_flagged[flag_columns].sum().astype(int),
        "share_of_rows": df_flagged[flag_columns].mean(),
    }
).sort_index()

## Zero Values

Zero values are concentrated in the transportation sector. They are retained as published values because the public EIA API does not expose observation-level reported/imputed/estimated flags.

In [ ]:
zero_by_sector = df_flagged.groupby("sectorid")[NUMERIC_COLUMNS].apply(
    lambda frame: (frame == 0).sum()
)

zero_by_sector

In [ ]:
df_flagged.loc[
    df_flagged["flag_all_metrics_zero"],
    ["period", "stateid", "stateDescription", "sectorid", "customers", "price", "revenue", "sales"],
].head(10)

## Negative Revenue Anomaly

A tiny negative revenue value was observed in the initial exploration. The value is retained and flagged rather than deleted because the cause is not identifiable from the public API alone.

In [ ]:
df_flagged.loc[
    df_flagged["flag_negative_revenue"],
    ["period", "stateid", "stateDescription", "sectorid", "customers", "price", "revenue", "sales"],
]

## Revenue Consistency Check

The relationship `revenue ~= sales * price / 100` is a reasonableness check, not a strict accounting identity. Published average price may be rounded, so small differences are expected.

In [ ]:
revenue_check_columns = [
    "period",
    "stateid",
    "sectorid",
    "revenue",
    "sales",
    "price",
    "expected_revenue",
    "revenue_diff",
    "revenue_abs_diff",
    "flag_revenue_consistency_check",
]

df_flagged.nlargest(10, "revenue_abs_diff")[revenue_check_columns]

In [ ]:
pd.Series(
    {
        "min_revenue_diff": df_flagged["revenue_diff"].min(),
        "max_revenue_diff": df_flagged["revenue_diff"].max(),
        "revenue_consistency_flags": int(df_flagged["flag_revenue_consistency_check"].sum()),
        "absolute_tolerance_million_usd": config["validation"]["revenue_abs_tolerance_million_usd"],
    }
)

## Validation Position

This dataset is ready for the next acquisition-stage review only if the structural checks pass and the quality flags are documented. It is not yet a license to make business claims, build final KPIs, or start modeling without a separate EDA and methodology step.

In [ ]:
assert validation_summary["duplicate_key_count"] == 0
assert validation_summary["row_count"] == config["validation"]["expected_rows"]
assert validation_summary["unique_months"] == config["validation"]["expected_months"]
assert validation_summary["unique_states"] == config["validation"]["expected_states"]
assert validation_summary["unique_sectors"] == config["validation"]["expected_sectors"]

print("Acquisition and validation notebook completed successfully.")